# Probes

> Probes attached to world models, to read quantities out of their latent space (stable-worldmodel API).

In [ ]:
#| default_exp wm.probes

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's world models package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from pathlib import Path

import torch
from torch import nn

In [ ]:
#| export
def attach_probe(model: nn.Module, key: str, probe: nn.Module):
    "Attach `probe` (e.g. a linear readout of the latent) to `model` under `key` (saved with the model)."
    assert isinstance(probe, nn.Module), 'Probe must be a nn.Module'
    if not hasattr(model, '_probes'):
        model._probes = nn.ModuleDict()
    model._probes[key] = probe


def get_probe(model: nn.Module, key: str) -> nn.Module | None:
    "The probe attached to `model` under `key`, or None."
    if hasattr(model, '_probes'):
        return model._probes[key] if key in model._probes else None
    return None


def load_probe(model: nn.Module, key: str, path: str | Path):
    "Load a probe saved at `path`: a whole module (attached under `key`) or the state dict of the probe under `key`."
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(payload, nn.Module):
        attach_probe(model, key, payload)
    elif isinstance(payload, dict):
        probe = get_probe(model, key)
        if probe is None:
            raise ValueError(f'No probe found for key {key} in model')
        probe.load_state_dict(payload)

### Tests

In [ ]:
import tempfile
from fastcore.test import test_fail
model = nn.Linear(4, 4)
assert get_probe(model, 'position') is None
probe = nn.Linear(4, 2)
attach_probe(model, 'position', probe)
assert get_probe(model, 'position') is probe and get_probe(model, 'other') is None
assert any(k.startswith('_probes.position') for k in model.state_dict())     # saved with the model
test_fail(lambda: attach_probe(model, 'x', lambda z: z), contains='nn.Module')
with tempfile.TemporaryDirectory() as tmp:
    other = nn.Linear(4, 2)
    torch.save(other.state_dict(), f'{tmp}/sd.pt')
    load_probe(model, 'position', f'{tmp}/sd.pt')                               # weights into the attached probe
    assert torch.equal(get_probe(model, 'position').weight, other.weight)
    torch.save(nn.Linear(4, 3), f'{tmp}/module.pt')
    load_probe(model, 'direction', f'{tmp}/module.pt')                         # a whole module
    assert get_probe(model, 'direction').out_features == 3
    test_fail(lambda: load_probe(nn.Linear(1, 1), 'position', f'{tmp}/sd.pt'), contains='No probe')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()